<a href="https://colab.research.google.com/github/labarboza14/Safe-Batch-Processing-with-REST-APIs/blob/main/Safe_Batch_Processing_with_REST_APIs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [26]:
!pip install -q pandas

In [27]:
import json
import time
import re
import pandas as pd
from datetime import datetime
from google.colab import files

# Tag fictícia (a "Risco Político" do seu case)
NOVA_TAG = "00000000-0000-0000-0000-000000000000"

EMAIL_COL = "email"
SLEEP_BETWEEN_REQUESTS_SEC = 0.05

# 🔥 Controle da demo:
# Se True, o mock do PUT vai FALHAR quando persona vier explicitamente como null
# Isso simula o comportamento "persona may not be null".
MOCK_PUT_FALHA_SE_PERSONA_NULL = True

def log(msg):
    agora = datetime.now().strftime("%H:%M:%S")
    print(f"[{agora}] {msg}")

In [28]:
EMAIL_RE = re.compile(r"^[^@\s]+@[^@\s]+\.[^@\s]+$", flags=re.IGNORECASE)

def is_valid_email(email):
    return isinstance(email, str) and bool(EMAIL_RE.match(email.strip()))

log("📤 Faça upload do CSV com coluna 'email'")
uploaded = files.upload()
if not uploaded:
    raise ValueError("Nenhum arquivo enviado.")

csv_path = next(iter(uploaded.keys()))
df = pd.read_csv(csv_path)

if EMAIL_COL not in df.columns:
    raise KeyError(f"Coluna '{EMAIL_COL}' não encontrada. Colunas: {list(df.columns)}")

emails_validos = [e.strip() for e in df[EMAIL_COL].astype(str) if is_valid_email(e)]

# dedupe case-insensitive preservando original
seen = set()
emails = []
for e in emails_validos:
    key = e.lower()
    if key not in seen:
        seen.add(key)
        emails.append(e)

log(f"📦 Total linhas: {len(df)}")
log(f"✉️ E-mails válidos: {len(emails_validos)}")
log(f"🔁 E-mails únicos (a processar): {len(emails)}")

[20:40:23] 📤 Faça upload do CSV com coluna 'email'


Saving Planilha sem título - Página1.csv to Planilha sem título - Página1 (2).csv
[20:40:33] 📦 Total linhas: 30
[20:40:33] ✉️ E-mails válidos: 30
[20:40:33] 🔁 E-mails únicos (a processar): 30


In [29]:
# Mock database: email -> contato
# Mistura proposital:
# - filters como dicts
# - filters como strings
# - persona ausente (None)
# - persona presente
# - alguns e-mails inexistentes no "banco"

def contato_mock(email, contato_id, filters, persona):
    return {
        "id": contato_id,
        "first_name": "Nome",
        "last_name": "Sobrenome",
        "email": email,
        "is_active": True,
        "filters": filters,
        "persona": persona,
    }

MOCK_DB = {}

for i, email in enumerate(emails[:25]):
    MOCK_DB[email] = {
        "id": f"c{i}",
        "first_name": "Nome",
        "last_name": "Sobrenome",
        "email": email,
        "is_active": True,
        "filters": ["tag_a", "tag_b"],
        "persona": None if i % 4 == 0 else {"id": "persona_x"}

}

# Para tornar a demo mais realista:
# - 20% dos e-mails serão "não encontrados" (se não estiverem no MOCK_DB)
# - 10% dos PUTs falharão por erro genérico (simulado)
import random
random.seed(7)

In [30]:
def normalize_filters(filters):
    """
    API pode retornar:
    - [{"id": "..."}]
    - ["uuid", ...]
    Converte para lista de strings UUID.
    """
    out = []
    for f in (filters or []):
        if isinstance(f, dict) and "id" in f:
            out.append(str(f["id"]))
        elif isinstance(f, str):
            out.append(f)

    # dedup mantendo ordem
    seen = set()
    dedup = []
    for x in out:
        if x not in seen:
            seen.add(x)
            dedup.append(x)
    return dedup

def normalize_persona(persona):
    if isinstance(persona, dict) and "id" in persona:
        return str(persona["id"])
    if isinstance(persona, str):
        return persona
    return None

def build_payload(contato, filtros_uuid):
    """
    ✅ Workaround do case: NÃO enviar persona se for None.
    """
    payload = {
        "first_name": contato.get("first_name"),
        "last_name": contato.get("last_name"),
        "email": contato.get("email"),
        "is_active": contato.get("is_active"),
        "filters": filtros_uuid,
    }

    persona_uuid = normalize_persona(contato.get("persona"))
    if persona_uuid is not None:
        payload["persona"] = persona_uuid  # só envia se existir

    return payload

In [31]:
class MockResponse:
    def __init__(self, status_code, data=None, text=""):
        self.status_code = status_code
        self._data = data
        self.text = text

    def json(self):
        return self._data

def mock_get_by_email(email):
    """
    Simula endpoint GET /contacts?email=
    Retorna {"count":..., "results":[...]} ou results vazio.
    """
    contato = MOCK_DB.get(email)
    if not contato:
        return MockResponse(200, {"count": 0, "results": []}, text="ok")
    return MockResponse(200, {"count": 1, "results": [contato]}, text="ok")

def mock_put_contact(contato_id, payload):
    """
    Simula PUT /contacts/{id}
    - Pode falhar se payload enviar "persona": null
    - Pode falhar aleatoriamente para simular instabilidade
    """
    # regra do case: persona null derruba
    if MOCK_PUT_FALHA_SE_PERSONA_NULL and ("persona" in payload and payload["persona"] is None):
        return MockResponse(400, data=None, text="persona may not be null")

    # falha aleatória (10%)
    if random.random() < 0.10:
        return MockResponse(500, data=None, text="temporary internal error")

    # se ok, atualiza o "banco" (simula persistência)
    # encontra contato pelo id
    for email, contato in MOCK_DB.items():
        if contato.get("id") == contato_id:
            # atualizar filters e persona se veio
            contato["filters"] = payload.get("filters", contato.get("filters"))
            if "persona" in payload:
                contato["persona"] = {"id": payload["persona"]} if isinstance(payload["persona"], str) else payload["persona"]
            return MockResponse(200, data={"ok": True}, text="ok")

    return MockResponse(404, data=None, text="not found")

In [32]:
sucessos = []
falhas = []
nao_encontrados = []

payloads_exemplo = []  # para mostrar depois (visual)

inicio = time.time()

for i, email in enumerate(emails, start=1):
    log(f"({i}/{len(emails)}) 🔎 {email}")

    # ---- GET
    r = mock_get_by_email(email)
    if r.status_code != 200:
        falhas.append({"email": email, "fase": "GET", "status": r.status_code, "erro": r.text})
        time.sleep(SLEEP_BETWEEN_REQUESTS_SEC)
        continue

    dados = r.json()
    results = dados.get("results") if isinstance(dados, dict) else None
    if not results:
        nao_encontrados.append(email)
        time.sleep(SLEEP_BETWEEN_REQUESTS_SEC)
        continue

    contato = results[0]
    contato_id = contato.get("id")
    if not contato_id:
        falhas.append({"email": email, "fase": "GET", "status": 200, "erro": "Contato sem id"})
        time.sleep(SLEEP_BETWEEN_REQUESTS_SEC)
        continue

    # ---- Normalizações
    filtros = normalize_filters(contato.get("filters"))

    # ---- Add tag
    if NOVA_TAG not in filtros:
        filtros.append(NOVA_TAG)

    # ---- Payload (com regra do case: não enviar persona se None)
    payload = build_payload(contato, filtros)

    # guarda alguns payloads para exibir
    if len(payloads_exemplo) < 5:
        payloads_exemplo.append({"email": email, "contato_id": contato_id, "payload": payload})

    # ---- PUT
    resp = mock_put_contact(contato_id, payload)

    if resp.status_code == 200:
        sucessos.append(email)
    else:
        falhas.append({"email": email, "fase": "PUT", "status": resp.status_code, "erro": resp.text, "payload": payload})

    time.sleep(SLEEP_BETWEEN_REQUESTS_SEC)

tempo_total = round(time.time() - inicio, 2)

# Export
pd.DataFrame({"email": sucessos}).to_csv("sucessos.csv", index=False)
pd.DataFrame(falhas).to_csv("falhas.csv", index=False)
pd.DataFrame({"email": nao_encontrados}).to_csv("nao_encontrados.csv", index=False)

log("📊 RESUMO FINAL")
log(f"✔ Sucessos: {len(sucessos)}")
log(f"❌ Falhas: {len(falhas)}")
log(f"⚠ Não encontrados: {len(nao_encontrados)}")
log(f"⏱ Tempo total: {tempo_total}s")

print("\n--- Exemplos de payload gerado (visual) ---")
for item in payloads_exemplo:
    print("\nEmail:", item["email"], "| contato_id:", item["contato_id"])
    print(json.dumps(item["payload"], indent=2, ensure_ascii=False))

[20:40:33] (1/30) 🔎 ana.fake@teste.com.br
[20:40:33] (2/30) 🔎 bruno.silva@exemplo.com
[20:40:33] (3/30) 🔎 carla.mendes@demo.com
[20:40:33] (4/30) 🔎 daniel.oliveira@sample.com
[20:40:33] (5/30) 🔎 eduarda.lima@testmail.com
[20:40:33] (6/30) 🔎 felipe.costa@mock.com
[20:40:33] (7/30) 🔎 gabriela.santos@fakemail.com
[20:40:33] (8/30) 🔎 henrique.alves@exemplo.com.br
[20:40:33] (9/30) 🔎 isabela.rocha@demo.org
[20:40:33] (10/30) 🔎 joao.pereira@sample.net
[20:40:33] (11/30) 🔎 karina.martins@testmail.org
[20:40:33] (12/30) 🔎 lucas.barros@mockmail.com
[20:40:34] (13/30) 🔎 mariana.freitas@demo.com.br
[20:40:34] (14/30) 🔎 nicolas.teixeira@sample.org
[20:40:34] (15/30) 🔎 olivia.ribeiro@fakemail.net
[20:40:34] (16/30) 🔎 paulo.fernandes@exemplo.com
[20:40:34] (17/30) 🔎 quezia.moraes@demo.com
[20:40:34] (18/30) 🔎 rafael.cardoso@testmail.com
[20:40:34] (19/30) 🔎 sara.barbosa@mock.org
[20:40:34] (20/30) 🔎 thiago.cunha@sample.com.br
[20:40:34] (21/30) 🔎 ursula.gomes@fakemail.com
[20:40:34] (22/30) 🔎 victor

In [33]:
files.download("sucessos.csv")
files.download("falhas.csv")
files.download("nao_encontrados.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>